In [16]:
# load ckpt : gpt2 small
from src.gapt import GPTConfig, GPT
import torch

gpt2_paths = {
    "small": "ckpt/fineweb10B-gpt2-small-20k.pt",
    "medium": "ckpt/fineweb10B-gpt2-medium-20k.pt",
    "large": "ckpt/fineweb10B-gpt2-large-20k.pt",
    "xl": "ckpt/fineweb10B-gpt2-xl-20k.pt",
}
gpt2_repo = {
    "small": "gpt2-small-fineweb10B",
    "medium": "gpt2-medium-fineweb10B",
    "large": "gpt2-large-fineweb10B",
    "xl": "gpt2-xl-fineweb10B"
}
gpt2_iblm_paths = {
    "small": "ckpt/fineweb10B-gpt2-small-softplus-spike.pt",
    "medium": "ckpt/fineweb10B-gpt2-medium-softplus-spike.pt",
    "large": "ckpt/fineweb10B-gpt2-large-softplus-spike.pt",
    "xl": "ckpt/fineweb10B-gpt2-xl-w40.pt",
}
gpt2_iblm_repo = {
    "small": "iblm-gpt2-small-fineweb10B",
    "medium": "iblm-gpt2-medium-fineweb10B",
    "large": "iblm-gpt2-large-fineweb10B",
    "xl": "iblm-gpt2-xl-fineweb10B"
}
gpt2_iblm2_paths = {
    "medium": "ckpt/fineweb10B-gpt2-medium-spike.pt"
}
gpt2_iblm2_repo = {
    "medium": "iblm2-gpt2-medium-fineweb10B"
}

model_size = "xl"
modgpt_config = GPTConfig.prior(name=model_size)
modgpt_model = GPT(modgpt_config)

# Load ckpt 
ckpt = torch.load(gpt2_iblm_paths[model_size], map_location="cpu")
state_dict = ckpt["model"]
# Fix for torch.compile: strip "_orig_mod." prefix if present
wanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(wanted_prefix):
        state_dict[k[len(wanted_prefix):]] = state_dict.pop(k)
modgpt_model.load_state_dict(state_dict)

<All keys matched successfully>

In [ ]:
# ----- Checking for porting error ------
# (1). Goal is to match CE loss, and generated token from local GPT and HF ported GPT
import torch
from src.gapt import GPT, GPTConfig

gpt_config = GPTConfig.prior(name="small")
gpt = GPT(gpt_config)

local_ckpt_path = "ckpt/fineweb10B-gpt2-small-20k.pt"
ckpt = torch.load(local_ckpt_path, map_location="cpu")
state_dict = ckpt["model"]
wanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(wanted_prefix):
        state_dict[k[len(wanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)

# ----- HF porting -----
from src.modeling_custom_gpt import CustomGPTConfig, CustomGPTModel, port_weights

hf_config = CustomGPTConfig(
    vocab_size=gpt_config.vocab_size,
    n_embd=gpt_config.n_embd,
    n_layer=gpt_config.n_layer,
    n_head=gpt_config.n_head,
    flex_kernel_options=gpt_config.flex_kernel_options
)
hf_model = CustomGPTModel(hf_config)

mapped_state = port_weights(gpt.state_dict(), hf_model.state_dict(), hf_config)
hf_model.load_state_dict(mapped_state, strict=True)

# logit matching, loss matching between 'gpt' and 'hf_model'
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer
from src.model import norm  # Import norm function

# IMPORTANT: Reload the module to pick up the port_weights fix!
import importlib
import src.modeling_custom_gpt
importlib.reload(src.modeling_custom_gpt)
from src.modeling_custom_gpt import port_weights

# Re-port the weights with the fixed function
mapped_state = port_weights(gpt.state_dict(), hf_model.state_dict(), hf_config)
hf_model.load_state_dict(mapped_state, strict=True)
print("✅ Weights re-ported with fixed port_weights function")

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Create test input
test_text = "The capital of France is Paris, and the weather there is"
input_ids = tokenizer.encode(test_text, return_tensors="pt")
target_ids = input_ids.clone()

print(f"Input shape: {input_ids.shape}")
print(f"Input tokens: {tokenizer.decode(input_ids[0])}")

# Set both models to eval mode
gpt.eval()
hf_model.eval()

with torch.no_grad():
    # ===== Local GPT model - compute logits directly =====
    # Run through transformer manually to get logits
    x = gpt.transformer.wte(input_ids)
    x = norm(x)
    x0 = x
    v1 = None
    
    # Simple causal mask for short sequence
    S = input_ids.shape[1]
    from src.model import create_block_mask
    docs = (input_ids == 50256).cumsum(1)
    def document_causal_mask(b, h, q_idx, kv_idx):
        causal_mask = q_idx >= kv_idx
        document_mask = docs[b, q_idx] == docs[b, kv_idx]
        return causal_mask & document_mask
    block_mask = create_block_mask(document_causal_mask, None, None, S, S, device="cpu", _compile=False)
    
    for layer in gpt.transformer.h:
        x, v1 = layer(x, v1, x0, block_mask)
    
    x = norm(x)
    local_logits = gpt.lm_head(x)
    local_logits = 30 * torch.tanh(local_logits / 30)
    local_logits = local_logits.float()
    
    # Compute local CE loss
    local_loss = F.cross_entropy(
        local_logits[:, :-1].contiguous().view(-1, local_logits.size(-1)),
        target_ids[:, 1:].contiguous().view(-1)
    )
    
    # ===== HF model =====
    hf_out = hf_model(input_ids, labels=target_ids)
    hf_logits = hf_out.logits
    hf_loss = hf_out.loss

print("\n" + "="*60)
print("LOGIT COMPARISON")
print("="*60)
print(f"Local logits shape: {local_logits.shape}")
print(f"HF logits shape:    {hf_logits.shape}")

# Compare logits
diff = (local_logits - hf_logits).abs()
print(f"\nMax logit diff:  {diff.max().item():.6f}")
print(f"Mean logit diff: {diff.mean().item():.6f}")
match_logits = diff.max().item() < 0.01
print(f"Logits match (< 0.01): {'✅ YES' if match_logits else '❌ NO'}")

print("\n" + "="*60)
print("LOSS COMPARISON")
print("="*60)
print(f"Local CE Loss: {local_loss.item():.6f}")
print(f"HF CE Loss:    {hf_loss.item():.6f}")
loss_diff = abs(local_loss.item() - hf_loss.item())
print(f"Loss diff:     {loss_diff:.6f}")
match_loss = loss_diff < 0.01
print(f"Loss match (< 0.01): {'✅ YES' if match_loss else '❌ NO'}")

print("\n" + "="*60)
print("NEXT TOKEN PREDICTION")
print("="*60)
local_next = local_logits[0, -1].argmax().item()
hf_next = hf_logits[0, -1].argmax().item()
print(f"Local predicts: '{tokenizer.decode([local_next])}' (id={local_next})")
print(f"HF predicts:    '{tokenizer.decode([hf_next])}' (id={hf_next})")
match_next = local_next == hf_next
print(f"Match: {'✅ YES' if match_next else '❌ NO'}")

print("\n" + "="*60)
print("OVERALL VERDICT")
print("="*60)
if match_logits and match_loss and match_next:
    print("✅ PORT SUCCESSFUL - Models are equivalent!")
else:
    print("❌ PORT FAILED - Models differ")

✅ Weights re-ported with fixed port_weights function
Input shape: torch.Size([1, 12])
Input tokens: The capital of France is Paris, and the weather there is

LOGIT COMPARISON
Local logits shape: torch.Size([1, 12, 50304])
HF logits shape:    torch.Size([1, 12, 50304])

Max logit diff:  0.000000
Mean logit diff: 0.000000
Logits match (< 0.01): ✅ YES

LOSS COMPARISON
Local CE Loss: 3.949824
HF CE Loss:    3.949824
Loss diff:     0.000000
Loss match (< 0.01): ✅ YES

NEXT TOKEN PREDICTION
Local predicts: ' very' (id=845)
HF predicts:    ' very' (id=845)
Match: ✅ YES

OVERALL VERDICT
✅ PORT SUCCESSFUL - Models are equivalent!


In [17]:
# hf_config = CustomGPTConfig(modgpt_config)
from src.modeling_custom_gpt import CustomGPTConfig, CustomGPTModel, port_weights

hf_config = CustomGPTConfig(
    vocab_size=modgpt_config.vocab_size,
    n_embd=modgpt_config.n_embd,
    n_layer=modgpt_config.n_layer,
    n_head=modgpt_config.n_head,
    flex_kernel_options=modgpt_config.flex_kernel_options
)
hf_model = CustomGPTModel(hf_config)

mapped_state = port_weights(modgpt_model.state_dict(), hf_model.state_dict(), hf_config)
hf_model.load_state_dict(mapped_state, strict=True)

<All keys matched successfully>

In [18]:
# Upload model checkpoint to HuggingFace Hub
# FORCE FRESH UPLOAD - clear cache and local dir first

from huggingface_hub import HfApi, login, create_repo, ModelCard
from transformers import AutoTokenizer
import shutil
import os

hf_usr_name = "Ksgk-fy"
hf_repo_name = gpt2_iblm_repo[model_size]
full_repo = f"{hf_usr_name}/{hf_repo_name}"

# 1. CLEAR LOCAL CACHE to force fresh save
local_dir = "./hf_ckpt"
if os.path.exists(local_dir):
    shutil.rmtree(local_dir)
os.makedirs(local_dir, exist_ok=True)

# 2. Create repo if doesn't exist
api = HfApi()
try:
    api.create_repo(full_repo, repo_type="model", exist_ok=True)
except Exception as e:
    print("Repo may already exist or there was an error. Proceeding...")

# 3. Print config for verification
print(f"Uploading model: {model_size}")
print(f"  n_embd:  {hf_config.n_embd}")
print(f"  n_layer: {hf_config.n_layer}")
print(f"  n_head:  {hf_config.n_head}")
print(f"  vocab:   {hf_config.vocab_size}")

# 4. Save model (this writes config.json with correct dimensions)
hf_model.save_pretrained(local_dir)
print(f"Saved to {local_dir}")

# 5. Copy modeling file
shutil.copy("src/modeling_custom_gpt.py", f"{local_dir}/modeling_custom_gpt.py")

# 6. Verify config.json is correct
import json
with open(f"{local_dir}/config.json", "r") as f:
    saved_config = json.load(f)
print(f"Saved config n_embd: {saved_config.get('n_embd')}")

# 7. Upload with unique commit message to force update
from huggingface_hub import upload_folder
import time

upload_folder(
    repo_id=full_repo,
    folder_path=local_dir,
    repo_type="model",
    commit_message=f"Force update config - n_embd={hf_config.n_embd} - {int(time.time())}"
)

print(f"\n✅ Uploaded to https://huggingface.co/{full_repo}")

# print(f"Model uploaded to https://huggingface.co/{full_repo}")


Uploading model: xl
  n_embd:  1600
  n_layer: 48
  n_head:  25
  vocab:   50304
Saved to ./hf_ckpt
Saved config n_embd: 1600


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


✅ Uploaded to https://huggingface.co/Ksgk-fy/iblm-gpt2-xl-fineweb10B


In [11]:
# gpt2 large ckpt's IBLM got more noticeable advantage here

In [14]:
# Test loading from HuggingFace - FORCE FRESH DOWNLOAD
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from huggingface_hub import snapshot_download
import shutil
import os

repo_id = f"Ksgk-fy/{gpt2_iblm_repo[model_size]}"
print(f"Loading: {repo_id}")

# Clear HF cache for this specific model to force fresh download
hf_cache = os.path.expanduser("~/.cache/huggingface/hub")
model_cache_pattern = repo_id.replace("/", "--")
for item in os.listdir(hf_cache) if os.path.exists(hf_cache) else []:
    if model_cache_pattern in item:
        cache_path = os.path.join(hf_cache, item)
        print(f"Clearing cache: {cache_path}")
        shutil.rmtree(cache_path, ignore_errors=True)

# Load fresh
model = AutoModelForCausalLM.from_pretrained(repo_id, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Verify config
print(f"\nLoaded model config:")
print(f"  n_embd:  {model.config.n_embd}")
print(f"  n_layer: {model.config.n_layer}")
print(f"  n_head:  {model.config.n_head}")

# Test generation
input_text = "Once upon a time"
input_ids = tokenizer.encode(input_text, return_tensors="pt")

with torch.no_grad():
    output_ids = model.generate(
        input_ids, 
        max_new_tokens=50, 
        do_sample=True, 
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )
print(f"\nGenerated: {tokenizer.decode(output_ids[0], skip_special_tokens=True)}")

Loading: Ksgk-fy/iblm-gpt2-xl-fineweb10B
Clearing cache: /Users/ksgk/.cache/huggingface/hub/models--Ksgk-fy--iblm-gpt2-xl-fineweb10B


config.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

modeling_custom_gpt.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Ksgk-fy/iblm-gpt2-xl-fineweb10B:
- modeling_custom_gpt.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.55G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/69.0 [00:00<?, ?B/s]


Loaded model config:
  n_embd:  1600
  n_layer: 48
  n_head:  25

Generated: Once upon a time.
And I’m going to say, when the people are gone, I’m going to be okay.”
–Leron Jeffers, “I’m Going to Be Okay”
The
